# 03 Image Regression Baseline

Image-to-scalar regression template based on the MNIST total-ink exercise.

In [ ]:
# Colab setup if needed
# from google.colab import drive
# drive.mount("/content/drive")
# !pip install -q -e .

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

from cnugs_ml.data import prepare_images, set_global_seed, describe_array
from cnugs_ml.models import build_image_regressor
from cnugs_ml.plots import plot_image_grid, plot_loss_history
from cnugs_ml.train import make_early_stopping
from cnugs_ml.submission import save_regression_submission

set_global_seed(42)

## Load data and build a scalar target

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

x_train = prepare_images(train_images, add_channel=True)
x_test = prepare_images(test_images, add_channel=True)

# Example target: total ink in the ORIGINAL 0-255 image.
y_train = np.sum(train_images, axis=(1, 2)).astype("float32")
y_test = np.sum(test_images, axis=(1, 2)).astype("float32")

describe_array("x_train", x_train)
describe_array("y_train", y_train)
plt.hist(y_train, bins=50)
plt.xlabel("Total ink")
plt.ylabel("Number of images")
plt.show()

## Build and train regressor

In [ ]:
model = build_image_regressor(
    input_shape=x_train.shape[1:],
    hidden_units=(64, 32),
    learning_rate=1e-3,
)
model.summary()

early_stopping = make_early_stopping(monitor="val_loss", patience=5)
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping],
)
plot_loss_history(history)

## Evaluate and visualize

In [ ]:
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f"test MSE = {test_loss:.3f}")
print(f"test MAE = {test_mae:.3f}")

pred = model.predict(x_test[:10]).reshape(-1)
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images[i], cmap="gray")
    plt.title(f"{pred[i]:.0f}/{y_test[i]:.0f}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Save predictions

In [ ]:
all_pred = model.predict(x_test).reshape(-1)
submission = save_regression_submission(all_pred, "../outputs/regression_submission.csv")
submission.head()